# Cleanup for Restart
This small notebook can be used to cleanup all outputs created by notebooks run after initial creation of the data set from s3.  The approach is workable because the notebooks generate stacks that are much much smaller than the full data set.  Further, those stacks were intentionally written to gridfs so the `delete_data` method used here will completely clean things out. 

First the stock incantation.

In [1]:
from mspasspy.db.database import Database
import mspasspy.client as msc
mspass_client = msc.Client()
dask_client = mspass_client.get_scheduler()
db = mspass_client.get_database("ES2026")

Stacks are all written to wf_TimeSeries with a data_tag.   This next box establishes all data tags. 

In [2]:
taglist=db.wf_TimeSeries.distinct("data_tag")
print(taglist)

['simple_stack', 'simple_stack_parallel', 'simple_stack_parallel_scatter_run', 'simple_stack_with_swp']


That confirms only stack data have tags.  Verify content.

In [3]:
for tag in taglist:
    query={"data_tag" : tag}
    nwf = db.wf_TimeSeries.count_documents(query)
    print(tag,nwf)

simple_stack 5
simple_stack_parallel 48
simple_stack_parallel_scatter_run 48
simple_stack_with_swp 48


Now the delete loop.

In [4]:
for tag in taglist:
    query={"data_tag" : tag}
    idlist=list()
    cursor = db.wf_TimeSeries.find(query)
    for doc in cursor:
        idlist.append(doc['_id'])
    print("Deleting ",len(idlist)," data with tag=",tag)
    for id in idlist:
        db.delete_data(id,"TimeSeries")

Deleting  5  data with tag= simple_stack
Deleting  48  data with tag= simple_stack_parallel
Deleting  48  data with tag= simple_stack_parallel_scatter_run
Deleting  48  data with tag= simple_stack_with_swp


Confirm that worked

In [5]:
for tag in taglist:
    query={"data_tag" : tag}
    nwf = db.wf_TimeSeries.count_documents(query)
    print(tag,nwf)

simple_stack 0
simple_stack_parallel 0
simple_stack_parallel_scatter_run 0
simple_stack_with_swp 0
